# Retrieval Augmented Translation (RAT): French to Target Language

This notebook implements a RAT pipeline using your parallel corpus.
- **Corpus**: `fr_kab.txt` (French-Target Language parallel sentences)
- **Embedding Model**: `BAAI/bge-m3`
- **Retriever**: `FAISS`
- **Translation Model**: `facebook/nllb-200-3.3B`

In [1]:
# 1. Install Dependencies
# Uncomment the line below to install required packages (CPU-only)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install transformers sentence-transformers faiss-cpu numpy

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached https://download.pytorch.org/whl/cpu/torchvision-0.24.1%2Bcpu-cp312-cp312-win_amd64.whl.metadata (6.1 kB)
  Using cached https://download.pytorch.org/whl/cpu/torchaudio-2.9.1%2Bcpu-cp312-cp312-win_amd64.whl.metadata (7.0 kB)
     ---------------------------------------- 0.0/536.2 kB ? eta -:--:--
     ---------------------------------------- 0.0/536.2 kB ? eta -:--:--
     ------------------- -------------------- 262.1/536.2 kB ? eta -:--:--
     ------------------- -------------------- 262.1/536.2 kB ? eta -:--:--
     ------------------------------------ 536.2/536.2 kB 605.0 kB/s eta 0:00:00
   ---------------------------------------- 0.0/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/110.9 MB 1.7 MB/s eta 0:01:06
   ---------------------------------------- 1.0/110.9 MB 1.7 MB/s eta 0:01:06
   --------------------------


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# First, try to import torch
try:
    import torch
    TORCH_AVAILABLE = True
    device = "cpu"
    print(f"PyTorch loaded successfully. Using device: {device}")
except OSError as e:
    TORCH_AVAILABLE = False
    device = "cpu"
    print(f"⚠️ PyTorch failed to load: {e}")
    print("Run this in terminal to fix:")
    print("  pip uninstall torch torchvision torchaudio -y")
    print("  pip install torch --index-url https://download.pytorch.org/whl/cpu")

import numpy as np
import faiss
import os
import pickle
from sentence_transformers import SentenceTransformer

# Only import transformers if torch is available
if TORCH_AVAILABLE:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Paths for saving/loading
INDEX_PATH = "./faiss_index.bin"
PAIRS_PATH = "./parallel_pairs.pkl"

OSError: [WinError 1114] Une routine d’initialisation d’une bibliothèque de liens dynamiques (DLL) a échoué. Error loading "d:\Documents\py\projects\chat_aqvayli\RAT\nevn\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# 2. Load Parallel Corpus from fr_kab.txt
corpus_path = "./fr_kab.txt"

french_sentences = []
Target Language_sentences = []

with open(corpus_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Split by tab(s) - the file uses tabs as separator
        parts = line.split('\t')
        # Filter out empty parts
        parts = [p.strip() for p in parts if p.strip()]
        if len(parts) >= 2:
            french_sentences.append(parts[0])
            Target Language_sentences.append(parts[1])

print(f"Loaded {len(french_sentences)} parallel sentence pairs.")
print(f"\nExample pairs:")
for i in range(min(3, len(french_sentences))):
    print(f"  FR: {french_sentences[i]}")
    print(f"  KAB: {Target Language_sentences[i]}\n")

In [ ]:
# 3. Initialize Embedding Model (BGE-M3)
# BGE-M3 is excellent for multilingual retrieval
embed_model_name = "BAAI/bge-m3"
embed_model = SentenceTransformer(embed_model_name, device=device)

print(f"Embedding model loaded: {embed_model_name}")

In [ ]:
# 4. Build or Load FAISS Vector Store

FORCE_REBUILD = False  # Set to True to rebuild even if saved files exist

if os.path.exists(INDEX_PATH) and os.path.exists(PAIRS_PATH) and not FORCE_REBUILD:
    # Load existing index and pairs
    print("Loading saved FAISS index and parallel pairs...")
    index = faiss.read_index(INDEX_PATH)
    with open(PAIRS_PATH, 'rb') as f:
        saved_data = pickle.load(f)
        parallel_pairs = saved_data['parallel_pairs']
        french_sentences = saved_data['french_sentences']
        Target Language_sentences = saved_data['Target Language_sentences']
    print(f"Loaded FAISS index with {index.ntotal} vectors.")
else:
    # Build new index
    print("Building new FAISS index...")

    # Load corpus if not already loaded
    if 'french_sentences' not in dir() or len(french_sentences) == 0:
        corpus_path = "./fr_kab.txt"
        french_sentences = []
        Target Language_sentences = []
        parallel_pairs = []

        with open(corpus_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split('\t')
                parts = [p.strip() for p in parts if p.strip()]
                if len(parts) >= 2:
                    french_sentences.append(parts[0])
                    Target Language_sentences.append(parts[1])
                    parallel_pairs.append({
                        "french": parts[0],
                        "Target Language": parts[1]
                    })

    # Generate embeddings
    print("Generating embeddings for French sentences...")
    french_embeddings = embed_model.encode(
        french_sentences,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    print(f"Embeddings shape: {french_embeddings.shape}")

    # Build FAISS index
    d = french_embeddings.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(french_embeddings.astype('float32'))

    # Save index and pairs
    print("Saving FAISS index and parallel pairs...")
    faiss.write_index(index, INDEX_PATH)
    with open(PAIRS_PATH, 'wb') as f:
        pickle.dump({
            'parallel_pairs': parallel_pairs,
            'french_sentences': french_sentences,
            'Target Language_sentences': Target Language_sentences
        }, f)
    print(f"Saved to {INDEX_PATH} and {PAIRS_PATH}")

print(f"FAISS index ready with {index.ntotal} vectors.")


def retrieve_examples(query_fr, k=5):
    """Retrieves top-k similar French sentences and their Target Language translations."""
    query_embedding = embed_model.encode(
        [query_fr], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = index.search(query_embedding.astype('float32'), k)

    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(parallel_pairs):
            results.append({
                "french": parallel_pairs[idx]["french"],
                "Target Language": parallel_pairs[idx]["Target Language"],
                "score": float(distances[0][i])
            })
    return results

In [ ]:
# 5. Initialize Translation Model (NLLB-200-3.3B)

if not TORCH_AVAILABLE:
    print("❌ Cannot load NLLB model - PyTorch is not available.")
    print("Please fix PyTorch installation first.")
else:
    model_name = "facebook/nllb-200-3.3B"
    print(f"Loading {model_name}... (this may take a while)")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    print("NLLB-200 initialized.")

    # Language codes for NLLB
    src_lang = "fra_Latn"  # French
    tgt_lang = "tgt_Latn"  # Target Language

Loading facebook/nllb-200-3.3B... (this may take a while)


NameError: name 'AutoTokenizer' is not defined

In [ ]:
# 6. RAT Translation Function
def translate_with_retrieval(text, num_examples=3, show_examples=True):
    """
    Translates French to Target Language using Retrieval Augmented Translation.

    1. Retrieve similar French-Target Language pairs from corpus
    2. Display retrieved examples as reference
    3. Translate using NLLB
    """
    # Step 1: Retrieve similar examples
    examples = retrieve_examples(text, k=num_examples)

    print(f"{'='*60}")
    print(f"INPUT (French): {text}")
    print(f"{'='*60}")

    if show_examples and examples:
        print(f"\n📚 Retrieved Translation Examples:")
        for i, ex in enumerate(examples, 1):
            print(f"  [{i}] (score: {ex['score']:.3f})")
            print(f"      FR:  {ex['french']}")
            print(f"      KAB: {ex['Target Language']}")
        print()

    # Step 2: Translate with NLLB
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)

    # Get target language token ID using convert_tokens_to_ids
    tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tgt_lang_id,
        max_length=256
    )

    translation = tokenizer.batch_decode(
        translated_tokens, skip_special_tokens=True)[0]

    print(f"🔄 NLLB Translation: {translation}")
    print(f"{'='*60}\n")

    return {
        "input": text,
        "translation": translation,
        "retrieved_examples": examples
    }

In [ ]:
# 7. Example Usage

# Example 1: Simple greeting
translate_with_retrieval("Bonjour, comment vas-tu ?")

# Example 2: Statement about language
translate_with_retrieval("La langue Target Language est très belle.")

# Example 3: Question
translate_with_retrieval("Où est ta maison ?")

In [ ]:
# 8. Interactive Translation (Optional)
def interactive_translate():
    """Run interactive translation session."""
    print("="*60)
    print("RAT French → Target Language Translator")
    print("Type 'quit' to exit")
    print("="*60)

    while True:
        user_input = input("\nEnter French text: ").strip()
        if user_input.lower() == 'quit':
            print("Goodbye!")
            break
        if user_input:
            translate_with_retrieval(user_input)

# Uncomment to run interactive mode:
# interactive_translate()

## How RAT Works Here

1. **Retrieval**: When you input a French sentence, BGE-M3 finds semantically similar French sentences from the corpus.

2. **Context**: The retrieved French-Target Language pairs serve as translation examples, showing how similar concepts are expressed in Target Language.

3. **Translation**: NLLB-200 translates the input. The retrieved examples help you evaluate and potentially improve the translation quality.

### Future Improvements
- Fine-tune NLLB on the parallel corpus
- Use retrieved examples in a prompt-based approach with a larger LLM
- Implement terminology extraction from retrieved examples